# Crash Severity Feature Analysis

This notebook demonstrates the full ML pipeline for crash severity prediction:
1. Data preprocessing
2. Feature selection using XGBoost and Random Forest
3. Class imbalance handling using SMOTEENN

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.preprocessing import preprocess_full_pipeline
from src.feature_selection import run_feature_selection
from src.resampling import run_resampling

In [2]:
# df = pd.read_csv("../data/US_Accidents_March23.csv")

In [3]:
# df.columns

In [4]:
# df.head(5)

In [5]:
# df['Astronomical_Twilight'].unique()

## Step 1: Data Preprocessing

Load and preprocess the US Accidents dataset.

In [6]:
X, y, config = preprocess_full_pipeline('../config.yaml')
print(f"Preprocessed data shape: X={X.shape}, y={y.shape}")
print(f"\nClass distribution:\n{pd.Series(y).value_counts().sort_index()}")

2026-03-15 16:08:54,505 - INFO - Loaded data: 5000 rows, 46 columns
2026-03-15 16:08:54,523 - INFO - Dropped columns: ['ID', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Description', 'Street', 'City', 'Country', 'Zipcode', 'Airport_Code', 'Weather_Timestamp', 'Source', 'County']
2026-03-15 16:08:54,526 - INFO - Converted 13 boolean columns to int
2026-03-15 16:08:54,536 - INFO - Missing values handled: 9968 -> 0 (9968 filled)
2026-03-15 16:08:54,545 - INFO - Label encoded 8 columns
2026-03-15 16:08:54,551 - INFO - Scaled 8 numerical features
2026-03-15 16:08:54,552 - INFO - Preprocessing complete: X shape (5000, 29), target distribution:
  Class 0: 4
  Class 1: 2938
  Class 2: 2053
  Class 3: 5


Preprocessed data shape: X=(5000, 29), y=(5000,)

Class distribution:
Severity
0       4
1    2938
2    2053
3       5
Name: count, dtype: int64


## Step 2: Feature Selection

Train XGBoost and Random Forest to identify top predictors.

In [7]:
selected_features = run_feature_selection(X, y, config)
print(f"\nSelected {len(selected_features)} common features:")
for f in selected_features:
    print(f"  - {f}")

2026-03-15 16:09:00,103 - INFO - Starting feature selection pipeline
2026-03-15 16:09:00,105 - INFO - Training XGBoost classifier...
2026-03-15 16:09:00,693 - INFO - XGBoost trained. Feature importances extracted.
2026-03-15 16:09:00,694 - INFO - Training Random Forest classifier...
2026-03-15 16:09:00,869 - INFO - Random Forest trained. Feature importances extracted.
2026-03-15 16:09:00,870 - INFO - Found 8 common features in top 12 from both models
2026-03-15 16:09:01,300 - INFO - Feature importance plot saved to /Users/pavanhebli/Personal/Projects/Ev Crash Project/outputs/figures/feature_importance_comparison.png
2026-03-15 16:09:01,301 - INFO - Feature selection complete. Selected 8 features



Selected 8 common features:
  - Distance(mi)
  - Temperature(F)
  - Wind_Direction
  - Wind_Speed(mph)
  - Crossing
  - Junction
  - Stop
  - Traffic_Signal


## Step 3: Class Imbalance Handling

Apply SMOTEENN to rebalance the dataset.

In [8]:
X_resampled, y_resampled = run_resampling(X[selected_features], y, config)
print(f"\nResampled data shape: X={X_resampled.shape}, y={y_resampled.shape}")

2026-03-15 16:09:40,376 - INFO - Starting resampling pipeline
2026-03-15 16:09:40,380 - INFO - Class distribution for 'Before SMOTEENN Resampling':
2026-03-15 16:09:40,381 - INFO -   Class 0: 4 (0.08%)
2026-03-15 16:09:40,382 - INFO -   Class 1: 2938 (58.76%)
2026-03-15 16:09:40,382 - INFO -   Class 2: 2053 (41.06%)
2026-03-15 16:09:40,383 - INFO -   Class 3: 5 (0.10%)
2026-03-15 16:09:40,398 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-15 16:09:40,399 - INFO - Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
2026-03-15 16:09:40,573 - INFO - Class distribution plot saved to /Users/pavanhebli/Personal/Projects/Ev Crash Project/outputs/figures/class_distribution_before_smoteenn_resampling.p


Resampled data shape: X=(7939, 8), y=(7939,)


## Summary

- Selected top features using ensemble agreement between XGBoost and RF
- Applied SMOTEENN to handle class imbalance
- Output plots saved to `outputs/figures/`